# Аудит журнала изменений модельной библиотеки

Цель: проверить, можно ли по `T_ENT_PARAM_CHG_HIST` восстановить журнал изменений моделей/версий: **что изменили → старое значение → новое значение → когда → кем**, а также найти аномальные паттерны изменений.

Источник основной логики: `T_ENT_PARAM_CHG_HIST`. По схеме в ней есть `ENT_SID`, `ENT_TYPE_NAME`, `ENT_PARAM_CHG_NAME`, `ENT_PARAM_CHG_PREV_VAL`, `ENT_PARAM_CHG_VAL`, `ENT_PARAM_CHG_TYPE_CODE`, `START_DTTM`, `END_DTTM`, `ENT_PARAM_CHG_USR_NAME`, `ENT_PARAM_CHG_USR_SID`, а также технические `VALIDFROM_DTTM/VALIDTO_DTTM`.

Важно: код сначала **проверяет фактические значения `ENT_TYPE_NAME`**, поэтому не предполагает заранее, что сущность называется `MODEL_VER`. Это принципиально: в схеме пример для `ENT_TYPE_NAME` — `MODEL`, а прямое соответствие `ENT_SID → MODEL_VER_SID` нужно подтвердить на данных. fileciteturn13file1L1-L12


In [ ]:
import os
import re
import sys
from typing import List

# Если Spark уже поднят в ноутбуке — переиспользуем его.
try:
    spark
except NameError:
    os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
    os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
    os.environ["PYSPARK_DRIVER"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
    sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

    from pyspark import SparkConf, SparkSession
    conf = (
        SparkConf()
        .setAppName("MODEL_LIBRARY_CHANGE_LOG_AUDIT")
        .setMaster("yarn")
        .set("spark.executor.cores", "2")
        .set("spark.executor.memory", "6g")
        .set("spark.executor.memoryOverhead", "1g")
        .set("spark.driver.memory", "6g")
        .set("spark.driver.maxResultSize", "1g")
        .set("spark.shuffle.service.enabled", "true")
        .set("spark.dynamicAllocation.enabled", "true")
        .set("spark.dynamicAllocation.initialExecutors", "3")
        .set("spark.dynamicAllocation.maxExecutors", "12")
        .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
        .set("spark.sql.session.timeZone", "Europe/Moscow")
    )
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

from pyspark.sql import functions as F, Window

SRC_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"

T_CHG_HIST = f"{SRC_DB}.T_ENT_PARAM_CHG_HIST"
T_CHG = f"{SRC_DB}.T_ENT_PARAM_CHG"
T_MODEL = f"{SRC_DB}.T_MODEL"
T_MODEL_VER = f"{SRC_DB}.T_MODEL_VER"
T_MODEL_VER_HIST = f"{SRC_DB}.T_MODEL_VER_HIST"
T_VALID = f"{SRC_DB}.T_VALID"

# Параметры анализа.
SHOW_USER_NAME = False       # True — показывать ФИО в результатах.
TOP_N = 50
BURST_PER_HOUR = 10          # эвристика: >= N изменений одним пользователем за час.
MASS_EDIT_PER_DAY = 10       # эвристика: >= N разных сущностей одним пользователем за день.
TOGGLE_THRESHOLD = 2         # эвристика: минимум N возвратов к предыдущему значению.
POST_VALIDATION_HOURS = 24   # окно для изменений после завершения валидации.

print("SOURCE:", SRC_DB)
print("Change log:", T_CHG_HIST)


## 1. Проверка таблиц и фактической схемы

Сначала не фильтруем `MODEL_VER`. Проверяем, какие типы сущностей реально присутствуют и какие поля доступны.

In [ ]:
def table_exists(full_name: str) -> bool:
    try:
        return bool(spark.catalog.tableExists(full_name))
    except Exception:
        try:
            spark.table(full_name).limit(1).count()
            return True
        except Exception:
            return False

for t in [T_CHG_HIST, T_CHG, T_MODEL, T_MODEL_VER, T_MODEL_VER_HIST, T_VALID]:
    print(f"{t}: {'OK' if table_exists(t) else 'MISSING'}")

if not table_exists(T_CHG_HIST):
    raise RuntimeError(f"Не найдена основная таблица журнала: {T_CHG_HIST}")

raw_hist = spark.table(T_CHG_HIST)
print("\nКолонки T_ENT_PARAM_CHG_HIST:")
print(raw_hist.columns)

raw_hist.printSchema()


## 2. Загрузка журнала

`START_DTTM` — бизнес-время изменения параметра; `VALIDFROM_DTTM/VALIDTO_DTTM` относятся к технической истории. Для аудита изменений используем бизнес-время и не отбрасываем исторические периоды.

Если один `ENT_PARAM_CHG_SID` встречается несколько раз в технической истории, оставляем последнюю техническую запись. Отдельно считаем такие повторы.

In [ ]:
def ts_col(df, name: str):
    raw = F.col(name)
    s = raw.cast("string")
    return F.coalesce(
        raw.cast("timestamp"),
        F.to_timestamp(s, "dd.MM.yyyy H:mm:ss"),
        F.to_timestamp(s, "dd.MM.yyyy HH:mm:ss"),
        F.to_timestamp(s, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
        F.to_timestamp(s, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(s, "yyyy-MM-dd"),
    )

def has_col(df, c: str) -> bool:
    return c.lower() in {x.lower() for x in df.columns}

def actual_col(df, c: str):
    mapping = {x.lower(): x for x in df.columns}
    return mapping.get(c.lower())

requested = [
    "ENT_PARAM_CHG_SID", "ENT_SID", "ENT_TYPE_NAME",
    "ENT_PARAM_CHG_NAME", "ENT_PARAM_CHG_TYPE_CODE",
    "ENT_PARAM_CHG_PREV_VAL", "ENT_PARAM_CHG_VAL",
    "START_DTTM", "END_DTTM",
    "ENT_PARAM_CHG_USR_NAME", "ENT_PARAM_CHG_USR_SID",
    "VALIDFROM_DTTM", "VALIDTO_DTTM",
]

missing = [c for c in requested if not has_col(raw_hist, c)]
print("Отсутствующие ожидаемые поля:", missing if missing else "нет")

exprs = []
for c in requested:
    ac = actual_col(raw_hist, c)
    if ac is None:
        exprs.append(F.lit(None).cast("string").alias(c.lower()))
    elif c in {"START_DTTM", "END_DTTM", "VALIDFROM_DTTM", "VALIDTO_DTTM"}:
        exprs.append(ts_col(raw_hist, ac).alias(c.lower()))
    else:
        exprs.append(F.col(ac).cast("string").alias(c.lower()))

chg_raw = raw_hist.select(*exprs)

# Диагностика технических дублей по ID изменения.
duplicate_ids = (
    chg_raw.groupBy("ent_param_chg_sid")
    .count()
    .filter(F.col("ent_param_chg_sid").isNotNull() & (F.col("count") > 1))
)

duplicate_id_summary = duplicate_ids.agg(
    F.count("*").alias("duplicate_change_ids"),
    F.sum("count").alias("rows_in_duplicate_groups"),
).withColumn(
    "extra_rows", F.col("rows_in_duplicate_groups") - F.col("duplicate_change_ids")
)
duplicate_id_summary.show(truncate=False)

# Дедупликация: последняя техническая версия записи.
w_sid = Window.partitionBy("ent_param_chg_sid").orderBy(
    F.col("validfrom_dttm").desc_nulls_last(),
    F.col("validto_dttm").desc_nulls_last(),
    F.col("start_dttm").desc_nulls_last(),
)

chg = (
    chg_raw
    .withColumn("_rn", F.row_number().over(w_sid))
    .filter((F.col("_rn") == 1) | F.col("ent_param_chg_sid").isNull())
    .drop("_rn")
    .withColumn("ent_type_norm", F.upper(F.trim(F.col("ent_type_name"))))
    .withColumn("param_norm", F.upper(F.trim(F.col("ent_param_chg_name"))))
    .withColumn("chg_date", F.to_date("start_dttm"))
    .withColumn("chg_hour", F.date_trunc("hour", "start_dttm"))
    .withColumn(
        "actor_key",
        F.when(
            F.length(F.trim(F.coalesce(F.col("ent_param_chg_usr_sid"), F.col("ent_param_chg_usr_name")))) > 0,
            F.sha2(F.trim(F.coalesce(F.col("ent_param_chg_usr_sid"), F.col("ent_param_chg_usr_name"))), 256),
        )
    )
)

print("Всего строк после дедупликации технической истории:", chg.count())


## 3. Какие сущности реально логируются

Это ключевая проверка. Если здесь присутствует тип вроде `MODEL_VER` / `MODEL_VERSION`, можно строить прямой version-level change log. Если присутствует только `MODEL`, журнал будет подтверждать изменения карточки модели, а не конкретной версии.

In [ ]:
entity_type_distribution = (
    chg.groupBy("ent_type_norm")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("ent_param_chg_name").alias("parameter_count"),
        F.countDistinct("actor_key").alias("actor_count"),
        F.min("start_dttm").alias("first_change"),
        F.max("start_dttm").alias("last_change"),
    )
    .orderBy(F.col("change_count").desc())
)

entity_type_distribution.show(TOP_N, truncate=False)

entity_types = [r["ent_type_norm"] for r in entity_type_distribution.collect() if r["ent_type_norm"]]

MODEL_VER_TYPES = [
    x for x in entity_types
    if "MODEL" in x and ("VER" in x or "VERSION" in x)
]
MODEL_TYPES = [
    x for x in entity_types
    if x == "MODEL" or ("MODEL" in x and "VER" not in x and "VERSION" not in x)
]

print("\nКандидаты на version-level сущность:", MODEL_VER_TYPES)
print("Кандидаты на model-level сущность:", MODEL_TYPES)


## 4. Что именно меняют

Здесь смотрим параметры, частоту изменений и переходы `старое → новое`. Это уже позволяет понять, является ли таблица пригодным журналом для change-management контроля.

In [ ]:
parameter_distribution = (
    chg.groupBy("ent_type_norm", "ent_param_chg_name")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("actor_key").alias("actor_count"),
        F.min("start_dttm").alias("first_change"),
        F.max("start_dttm").alias("last_change"),
    )
    .orderBy(F.col("change_count").desc())
)

parameter_distribution.show(TOP_N, truncate=False)

transitions = (
    chg.filter(
        F.col("ent_param_chg_prev_val").isNotNull()
        | F.col("ent_param_chg_val").isNotNull()
    )
    .groupBy("ent_type_norm", "ent_param_chg_name", "ent_param_chg_prev_val", "ent_param_chg_val")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("actor_key").alias("actor_count"),
    )
    .orderBy(F.col("change_count").desc())
)

print("Топ переходов старое → новое:")
transitions.show(TOP_N, truncate=False)


## 5. Покрытие пользователей и активности

Проверяем, действительно ли журнал позволяет установить автора изменения. Для анализа поведения пользователя используем хеш `actor_key`; исходный `ENT_PARAM_CHG_USR_NAME` можно включить параметром `SHOW_USER_NAME=True`.

In [ ]:
user_coverage = chg.agg(
    F.count("*").alias("changes"),
    F.sum(F.when(F.col("ent_param_chg_usr_sid").isNotNull() & (F.trim(F.col("ent_param_chg_usr_sid")) != ""), 1).otherwise(0)).alias("with_user_sid"),
    F.sum(F.when(F.col("ent_param_chg_usr_name").isNotNull() & (F.trim(F.col("ent_param_chg_usr_name")) != ""), 1).otherwise(0)).alias("with_user_name"),
    F.countDistinct("actor_key").alias("distinct_actors"),
).withColumn("user_sid_fill_pct", 100 * F.col("with_user_sid") / F.col("changes"))\
 .withColumn("user_name_fill_pct", 100 * F.col("with_user_name") / F.col("changes"))

user_coverage.show(truncate=False)

actor_distribution = (
    chg.filter(F.col("actor_key").isNotNull())
    .groupBy("actor_key")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("ent_type_norm").alias("entity_type_count"),
        F.countDistinct("ent_param_chg_name").alias("parameter_count"),
        F.min("start_dttm").alias("first_change"),
        F.max("start_dttm").alias("last_change"),
    )
    .orderBy(F.col("change_count").desc())
)

actor_distribution.show(TOP_N, truncate=False)


## 6. Проверка корректности самого журнала

Контроли качества данных:
- `old = new` — изменение фактически ничего не изменило;
- нет времени изменения;
- нет автора;
- `END_DTTM < START_DTTM`;
- повторный `ENT_PARAM_CHG_SID` в технической истории.

In [ ]:
quality_checks = [
    ("no_op_change", chg.filter(
        F.col("ent_param_chg_prev_val").isNotNull()
        & F.col("ent_param_chg_val").isNotNull()
        & (F.trim(F.col("ent_param_chg_prev_val")) == F.trim(F.col("ent_param_chg_val")))
    )),
    ("missing_change_time", chg.filter(F.col("start_dttm").isNull())),
    ("missing_actor", chg.filter(F.col("actor_key").isNull())),
    ("end_before_start", chg.filter(
        F.col("start_dttm").isNotNull() & F.col("end_dttm").isNotNull()
        & (F.col("end_dttm") < F.col("start_dttm"))
    )),
]

quality_rows = []
for name, df in quality_checks:
    n = df.count()
    quality_rows.append((name, n))

spark.createDataFrame(quality_rows, ["check", "flagged_rows"]).show(truncate=False)
print("Технических duplicate-ID групп:", duplicate_ids.count())


## 7. Поиск аномальной активности пользователей

Эти правила являются **аналитическими эвристиками**, а не утверждениями из схемы. Пороговые значения вынесены в начало ноутбука и должны быть откалиброваны по фактическому распределению.

In [ ]:
hourly_burst = (
    chg.filter(F.col("actor_key").isNotNull() & F.col("chg_hour").isNotNull())
    .groupBy("actor_key", "chg_hour")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("ent_param_chg_name").alias("parameter_count"),
    )
    .filter(F.col("change_count") >= BURST_PER_HOUR)
    .orderBy(F.col("change_count").desc())
)

mass_edit = (
    chg.filter(F.col("actor_key").isNotNull() & F.col("chg_date").isNotNull())
    .groupBy("actor_key", "chg_date")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("ent_param_chg_name").alias("parameter_count"),
    )
    .filter(F.col("entity_count") >= MASS_EDIT_PER_DAY)
    .orderBy(F.col("entity_count").desc(), F.col("change_count").desc())
)

print(f"Часовые всплески >= {BURST_PER_HOUR} изменений:")
hourly_burst.show(TOP_N, truncate=False)

print(f"Массовые изменения >= {MASS_EDIT_PER_DAY} разных сущностей в день:")
mass_edit.show(TOP_N, truncate=False)


## 8. Повторные переключения параметров

Ищем последовательности вида `A → B → A`. Для статусов и других критичных параметров это может быть полезным индикатором возвратов/откатов.

In [ ]:
w_entity_param = Window.partitionBy("ent_type_norm", "ent_sid", "ent_param_chg_name").orderBy(
    F.col("start_dttm").asc_nulls_last(),
    F.col("ent_param_chg_sid").asc_nulls_last(),
)

toggle_log = (
    chg.withColumn("prev_new_val", F.lag("ent_param_chg_val").over(w_entity_param))
       .withColumn("prev_old_val", F.lag("ent_param_chg_prev_val").over(w_entity_param))
       .filter(
           F.col("ent_param_chg_val").isNotNull()
           & F.col("prev_old_val").isNotNull()
           & (F.trim(F.col("ent_param_chg_val")) == F.trim(F.col("prev_old_val")))
       )
       .select(
           "ent_type_norm", "ent_sid", "ent_param_chg_name", "start_dttm",
           "ent_param_chg_prev_val", "ent_param_chg_val",
           "prev_new_val", "prev_old_val", "actor_key",
       )
       .orderBy(F.col("start_dttm").desc_nulls_last())
)

print("Найденные возвраты к предыдущему значению:")
toggle_log.show(TOP_N, truncate=False)


## 9. Version-level change log

Если `MODEL_VER_TYPES` найден — строим полноценный журнал по версиям и присоединяем текущие атрибуты версии из `T_MODEL_VER`.

Если `MODEL_VER_TYPES` пуст — **не подменяем модельную сущность версионной**. Вместо этого отдельно показываем model-level журнал. Это позволяет честно определить ограничение витрины.

In [ ]:
def decorate_actor(df):
    if SHOW_USER_NAME:
        return df.withColumn("actor_display", F.coalesce(F.col("ent_param_chg_usr_name"), F.col("actor_key")))
    return df.withColumn("actor_display", F.col("actor_key"))

model_ver_log = None
model_log = None

if MODEL_VER_TYPES:
    mv = spark.table(T_MODEL_VER)
    mv_cols = {c.upper(): c for c in mv.columns}
    required_mv = ["MODEL_VER_SID", "MODEL_SID", "MODEL_VER_NUM", "MODEL_VER_STTS_NAME", "MODEL_VER_SIGNFCNT_CTGRY_CODE"]
    available_mv = [mv_cols[c] for c in required_mv if c in mv_cols]

    mv_meta = mv.select(*[F.col(c) for c in available_mv]).dropDuplicates([mv_cols["MODEL_VER_SID"]])
    mv_meta = mv_meta.withColumnRenamed(mv_cols["MODEL_VER_SID"], "mv_join_sid")

    model_ver_log = (
        decorate_actor(
            chg.filter(F.col("ent_type_norm").isin(MODEL_VER_TYPES))
        )
        .withColumn("model_ver_sid", F.col("ent_sid"))
        .join(mv_meta, F.col("model_ver_sid") == F.col("mv_join_sid"), "left")
        .drop("mv_join_sid")
        .select(
            "model_ver_sid", "model_sid", "model_ver_num",
            "model_ver_stts_name", "model_ver_signfcnt_ctgry_code",
            "start_dttm", "end_dttm",
            "actor_display", "ent_param_chg_usr_sid",
            "ent_param_chg_name", "ent_param_chg_prev_val", "ent_param_chg_val",
            "ent_param_chg_type_code", "ent_param_chg_sid",
        )
        .orderBy(F.col("start_dttm").desc_nulls_last())
    )

    print("Version-level журнал:")
    model_ver_log.show(TOP_N, truncate=False)

else:
    print("Прямой MODEL_VER entity type не найден.")
    print("Проверяем model-level журнал, но не называем его version-level.")

    if MODEL_TYPES:
        model_log = (
            decorate_actor(chg.filter(F.col("ent_type_norm").isin(MODEL_TYPES)))
            .withColumn("model_sid", F.col("ent_sid"))
            .select(
                "model_sid", "start_dttm", "end_dttm",
                "actor_display", "ent_param_chg_usr_sid",
                "ent_param_chg_name", "ent_param_chg_prev_val", "ent_param_chg_val",
                "ent_param_chg_type_code", "ent_param_chg_sid",
            )
            .orderBy(F.col("start_dttm").desc_nulls_last())
        )
        model_log.show(TOP_N, truncate=False)


## 10. Изменения после валидации

Если version-level связь подтверждена, проверяем изменения параметров в течение `POST_VALIDATION_HOURS` после последней завершённой валидации, предшествующей изменению.

Это уже полезный change-management контроль: изменение критичного параметра после контрольной процедуры.

In [ ]:
post_validation_changes = None

if model_ver_log is not None and table_exists(T_VALID):
    valid = spark.table(T_VALID)
    vc = {c.upper(): c for c in valid.columns}
    if all(x in vc for x in ["MODEL_VER_SID", "VALID_END_FACT_DTTM"]):
        valid_end = (
            valid.select(
                F.col(vc["MODEL_VER_SID"]).cast("string").alias("v_model_ver_sid"),
                F.col(vc["VALID_END_FACT_DTTM"]).cast("timestamp").alias("valid_end_dttm"),
            )
            .filter(F.col("valid_end_dttm").isNotNull())
            .dropDuplicates()
        )

        c = model_ver_log.alias("c")
        v = valid_end.alias("v")

        joined = (
            c.join(
                v,
                (F.col("c.model_ver_sid") == F.col("v.v_model_ver_sid"))
                & (F.col("v.valid_end_dttm") <= F.col("c.start_dttm")),
                "left",
            )
            .withColumn(
                "hours_after_validation",
                (F.unix_timestamp("c.start_dttm") - F.unix_timestamp("v.valid_end_dttm")) / 3600.0,
            )
        )

        w_nearest = Window.partitionBy("c.ent_param_chg_sid").orderBy(F.col("v.valid_end_dttm").desc_nulls_last())

        post_validation_changes = (
            joined.withColumn("_rn", F.row_number().over(w_nearest))
            .filter(
                (F.col("_rn") == 1)
                & F.col("v.valid_end_dttm").isNotNull()
                & (F.col("hours_after_validation") >= 0)
                & (F.col("hours_after_validation") <= POST_VALIDATION_HOURS)
            )
            .drop("_rn")
            .select(
                "c.model_ver_sid", "c.model_ver_num", "c.start_dttm",
                "v.valid_end_dttm", "hours_after_validation",
                "c.actor_display", "c.ent_param_chg_name",
                "c.ent_param_chg_prev_val", "c.ent_param_chg_val",
                "c.ent_param_chg_type_code", "c.ent_param_chg_sid",
            )
            .orderBy(F.col("c.start_dttm").desc_nulls_last())
        )

        print(f"Изменения в течение {POST_VALIDATION_HOURS} ч после завершённой валидации:")
        post_validation_changes.show(TOP_N, truncate=False)
    else:
        print("T_VALID есть, но не найдена пара MODEL_VER_SID + VALID_END_FACT_DTTM; контроль пропущен.")
else:
    print("Post-validation контроль пропущен: нет подтверждённого version-level журнала или T_VALID.")


## 11. Критичные параметры

Список ниже — настраиваемая эвристика для аудита. Он не утверждает, что именно эти поля являются нормативно критичными. Используется для поиска изменений, которые стоит вручную проверить.

In [ ]:
CRITICAL_PARAM_RE = (
    r"STATUS|СТАТУС|TARGET|TARGET|МЕТОД|METHOD|REPOSIT|РЕПОЗ|COMMIT|"
    r"SIGNIFIC|ЗНАЧИМ|RISK|РИСК|VALID|ВАЛИД|PROD|ПРОМЫШ|OWNER|ВЛАД|"
    r"SAMPLE|ВЫБОР|DATA|ДАНН|FEATURE|ФИЧ|PROMPT|ПРОМПТ|LLM"
)

critical_changes = chg.filter(F.col("param_norm").rlike(CRITICAL_PARAM_RE))

critical_summary = (
    critical_changes.groupBy("ent_type_norm", "ent_param_chg_name")
    .agg(
        F.count("*").alias("change_count"),
        F.countDistinct("ent_sid").alias("entity_count"),
        F.countDistinct("actor_key").alias("actor_count"),
        F.min("start_dttm").alias("first_change"),
        F.max("start_dttm").alias("last_change"),
    )
    .orderBy(F.col("change_count").desc())
)

critical_summary.show(TOP_N, truncate=False)


## 12. Итоговая матрица пригодности журнала

Результат должен отвечать на четыре вопроса:

1. Есть ли в данных version-level сущность?
2. Можно ли установить автора изменения?
3. Есть ли старое и новое значение?
4. Есть ли корректное бизнес-время изменения?

Если все четыре пункта подтверждены, `T_ENT_PARAM_CHG_HIST` можно использовать как основной источник change-management контроля. `T_MODEL_VER_HIST` при этом остаётся дополнительным источником для восстановления состояния версии во времени.

In [ ]:
total_changes = chg.count()
with_old = chg.filter(F.col("ent_param_chg_prev_val").isNotNull()).count()
with_new = chg.filter(F.col("ent_param_chg_val").isNotNull()).count()
with_time = chg.filter(F.col("start_dttm").isNotNull()).count()
with_actor = chg.filter(F.col("actor_key").isNotNull()).count()

summary = [
    ("version_level_entity_present", bool(MODEL_VER_TYPES), "types=" + ", ".join(MODEL_VER_TYPES)),
    ("old_value_available_pct", 100.0 * with_old / total_changes if total_changes else None, "ENT_PARAM_CHG_PREV_VAL"),
    ("new_value_available_pct", 100.0 * with_new / total_changes if total_changes else None, "ENT_PARAM_CHG_VAL"),
    ("change_time_available_pct", 100.0 * with_time / total_changes if total_changes else None, "START_DTTM"),
    ("actor_available_pct", 100.0 * with_actor / total_changes if total_changes else None, "ENT_PARAM_CHG_USR_SID/NAME"),
    ("technical_duplicate_change_ids", duplicate_ids.count(), "ENT_PARAM_CHG_SID"),
]

spark.createDataFrame(summary, ["check", "value", "source_field"]).show(truncate=False)


## 13. Опционально: выгрузка ключевых результатов

По умолчанию запись в витрину выключена. При необходимости можно сохранить результаты как managed tables в `OUT_DB`.

In [ ]:
SAVE_RESULTS = False
OUT_DB = "arnsdpsbx_t_team_oam_sva_2"
PREFIX = "model_change_audit"

if SAVE_RESULTS:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {OUT_DB}")
    entity_type_distribution.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_entity_types")
    parameter_distribution.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_parameters")
    transitions.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_transitions")
    actor_distribution.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_actors")
    hourly_burst.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_hourly_burst")
    mass_edit.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_mass_edit")
    toggle_log.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_toggles")
    critical_summary.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_critical_parameters")
    if model_ver_log is not None:
        model_ver_log.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_version_log")
    if post_validation_changes is not None:
        post_validation_changes.write.mode("overwrite").saveAsTable(f"{OUT_DB}.{PREFIX}_post_validation")
    print("Результаты сохранены в", OUT_DB)
else:
    print("SAVE_RESULTS=False — таблицы не создаются.")
